# Complainify AI — Complete Study Notebook

A self-contained, **from-scratch** walkthrough of the NLP engine that powers complaint processing:

1. Dataset structure (10 categories)
2. Text preprocessing - stopwords, custom stemmer, bigrams
3. 80/20 train/test split on the real `data/train_dataset.csv`
4. Multinomial Naive Bayes classifier (written from scratch)
5. Evaluation metrics (accuracy, precision, recall, F1)
6. Sentiment analysis (lexicon-based, negation + intensifiers)
7. New-word handling in sentiment (unknown words, synonyms)
8. Complete sentiment → priority pipeline (mirrors `ml/priority.py`)
9. Real model performance from the live `ml/training_log.json`
10. Live status (matches the admin dashboard)

> No ML libraries required — every algorithm is pure Python. The only data
> dependencies are `data/train_dataset.csv` and `ml/training_log.json` from
> the project folder. Cells probe several candidate paths, so the notebook
> runs no matter which working directory it is opened from.

---
## PART 1: DATASET STRUCTURE

Complaints belong to **10 categories**. Production code loads the same mapping
from `data/encoders/category_decoder.json`. Here it is, with the numeric
`category_encoded` IDs used by the CSV.

In [ ]:
import json, csv, os, math, re, random
from collections import Counter, defaultdict

CAT_DECODER = {
    0: 'IT Support', 1: 'Hostels', 2: 'Academics', 3: 'Fees / Finance',
    4: 'Maintenance', 5: 'Transport', 6: 'Security / Discipline',
    7: 'Administration', 8: 'Library', 9: 'Canteen'
}

print(f'{len(CAT_DECODER)} categories:')
for cid, cname in CAT_DECODER.items():
    print(f'  {cid}: {cname}')

---
## PART 2: TEXT PREPROCESSING PIPELINE

Three steps that turn raw text into model features:

- **Stopword removal** — drop ~80 high-frequency, low-signal words
- **Custom stemmer** — 30+ suffix rules: `working / worked / worker → work`
- **Bigrams** — join adjacent tokens with `_` so phrases like `not_working`
  survive as one feature

> Note: the *classifier* removes stopwords (including `not`), but *sentiment
> analysis* must keep them — `not working` means the opposite of `working`.

In [ ]:
STOPWORDS = set("""
    a an the is are was were be been being have has had do does did
    will would shall should may might must can could of in on at by
    for with about against between into through during before after
    above below to from up down out off over under again further then
    once here there when where why how all each every both few more
    most other some such no nor not only own same so than too very
    just because as until while
""".split())
print(f'Stopwords loaded: {len(STOPWORDS)} words')

In [ ]:
def stem(w):
    """Custom rule-based stemmer — no external library needed."""
    if len(w) < 5:
        return w
    # Longer suffixes first — order matters!
    if w.endswith('ingly'): return w[:-5]
    if w.endswith('edly'):  return w[:-4] + 'y'
    if w.endswith('ying'):  return w[:-4] + 'y'
    if w.endswith('ation'): return w[:-5]
    if w.endswith('ment'):  return w[:-4]
    if w.endswith('able'): return w[:-4]
    if w.endswith('ible'): return w[:-4]
    if w.endswith('ness'): return w[:-4]
    if w.endswith('less'): return w[:-4]
    if w.endswith('ally'): return w[:-4]
    if w.endswith('sion'): return w[:-3] + 's'
    if w.endswith('tion'): return w[:-3] + 't'
    if w.endswith('ical'): return w[:-4]
    if w.endswith('ied'):  return w[:-3] + 'y'
    if w.endswith('ies'):  return w[:-3] + 'y'
    if w.endswith('ing'):  return w[:-3]
    if w.endswith('ive'):  return w[:-3]
    if w.endswith('ful'):  return w[:-3]
    if w.endswith('ous'):  return w[:-3]
    if w.endswith('ise'):  return w[:-3]
    if w.endswith('ize'):  return w[:-3]
    if w.endswith('ate'):  return w[:-3]
    if w.endswith('ify'):  return w[:-3]
    if w.endswith('ed'):   return w[:-2]
    if w.endswith('er'):   return w[:-2]
    if w.endswith('or'):   return w[:-2]
    if w.endswith('ly'):   return w[:-2]
    if w.endswith('al'):   return w[:-2]
    if w.endswith('en'):   return w[:-2]
    if w.endswith('s') and not w.endswith('ss'):
        return w[:-1]
    return w

test_words = ['working', 'worked', 'worker', 'satisfying', 'organization',
              'happiness', 'tension', 'action', 'books', 'class', 'quickly']
for w in test_words:
    print(f'  {w:15s} -> {stem(w):12s}')

In [ ]:
def clean_and_tokenize(text, add_bigrams=True):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = [stem(t) for t in text.split()
              if t not in STOPWORDS and len(t) > 2]
    if add_bigrams and len(tokens) > 1:
        tokens += ['_'.join(pair) for pair in zip(tokens, tokens[1:])]
    return tokens

example = "The WiFi is not working in the library!"
toks = clean_and_tokenize(example)
print(f'Original : "{example}"')
print(f'Tokens   : {toks}')
print(f'Bigrams  : {[t for t in toks if "_" in t]}')
print()
print('Breakdown:')
print('  "the"     -> removed (stopword)')
print('  "wifi"    -> kept')
print('  "is" / "in" -> removed (stopword)')
print('  "not"     -> removed (stopword - classifier side only, see PART 6)')
print('  "working" -> stemmed to "work"')
print('  "library" -> stemmed to "librari"')
print('  "!"       -> removed (punctuation regex)')

---
## PART 3: LOAD & SPLIT DATA (80/20)

Same flow as production `retrain.py`: read `data/train_dataset.csv`, map
category names to numeric IDs, shuffle with a fixed seed, split 80/20.

The path probe below tries several candidate locations so the cell works
regardless of the notebook's working directory.

In [ ]:
DATA_CANDIDATES = [
    'data/train_dataset.csv',
    '../data/train_dataset.csv',
    '../../data/train_dataset.csv',
    '../../../data/train_dataset.csv',
    r'E:/Project-VI/workspace/ComplaintMgmtSystem/data/train_dataset.csv',
]
TRAIN_PATH = next((p for p in DATA_CANDIDATES if os.path.isfile(p)), None)
if TRAIN_PATH is None:
    raise FileNotFoundError('train_dataset.csv not found - checked candidates:')
print('Using dataset :', TRAIN_PATH)

rows = list(csv.DictReader(open(TRAIN_PATH, encoding='utf-8')))
print('Total rows    :', len(rows))
print('Sample row    :', dict(list(rows[0].items())))

counts = Counter(r['category'] for r in rows)
print('Category distribution:')
for cat, n in counts.most_common():
    print(f'  {cat:<22} {n:>5}  ({n / len(rows) * 100:.1f}%)')

CAT_NAME_TO_ID = {v: k for k, v in CAT_DECODER.items()}
encoded = [{'text': r['text'], 'category_encoded': CAT_NAME_TO_ID[r['category']]}
           for r in rows if r['category'] in CAT_NAME_TO_ID]

random.seed(42)
random.shuffle(encoded)
split = int(len(encoded) * 0.8)
train_rows, test_rows = encoded[:split], encoded[split:]
print('80/20 split : train', len(train_rows), '/ test', len(test_rows))

sample_data = random.sample(encoded, 200)
print('sample_data (200 rows, for teaching below):')
for s in sample_data[:4]:
    print('  ', s)

---
## PART 4: MULTINOMIAL NAIVE BAYES — FROM SCRATCH

For a document `d` and class `c` we estimate:

- **Prior**      `P(c) = docs_in_c / total_docs`
- **Likelihood** `P(w|c) = (count(w,c) + alpha) / (total_words_c + alpha * |V|)`   *(Laplace smoothing)*
- **Score**      `log P(c|d) ~ log P(c) + sum(log P(w|c))`
- **Softmax**    converts the scores into probabilities that sum to 1.0


In [ ]:
class MultinomialNB:
    """Multinomial Naive Bayes from scratch (JSON-serialisable)."""

    def __init__(self, alpha=1.0, min_df=3):
        self.alpha = alpha
        self.min_df = min_df
        self._trained = False

    def fit(self, texts, labels):
        self.classes = sorted(set(labels))
        n = len(texts)
        class_docs = Counter(labels)
        self.priors = {c: math.log(class_docs[c] / n) for c in self.classes}

        all_tokenized = [clean_and_tokenize(t) for t in texts]
        doc_freq = Counter()
        for tokens in all_tokenized:
            for token in set(tokens):
                doc_freq[token] += 1
        self.vocab = {w for w, f in doc_freq.items() if f >= self.min_df}
        self.vocab_size = len(self.vocab)

        self.word_counts = {c: defaultdict(int) for c in self.classes}
        self.class_total_words = {c: 0 for c in self.classes}
        for tokens, label in zip(all_tokenized, labels):
            for token in set(tokens):
                if token in self.vocab:
                    self.word_counts[label][token] += 1
                    self.class_total_words[label] += 1
        self._trained = True
        print(f'Model trained: {len(self.classes)} classes, {self.vocab_size} vocab words')

    def predict_with_proba(self, text):
        tokens = clean_and_tokenize(text)
        scores = {}
        for c in self.classes:
            log_prob = self.priors[c]
            total_wc = self.class_total_words[c]
            for token in tokens:
                count = self.word_counts[c].get(token, 0)
                log_prob += math.log((count + self.alpha) /
                                     (total_wc + self.alpha * self.vocab_size))
            scores[c] = log_prob
        best = max(scores, key=scores.get)
        log_vals = list(scores.values())
        max_log = max(log_vals)
        exp_vals = [math.exp(v - max_log) for v in log_vals]
        total = sum(exp_vals)
        probs = {c: exp / total for c, exp in zip(scores.keys(), exp_vals)}
        return best, probs

    def save(self, path):
        data = {
            'alpha': self.alpha, 'min_df': self.min_df,
            'classes': self.classes, 'priors': self.priors,
            'vocab': list(self.vocab), 'vocab_size': self.vocab_size,
            'class_total_words': self.class_total_words,
            'word_counts': {str(c): dict(wc) for c, wc in self.word_counts.items()}
        }
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f'Model saved to {path}')

    @classmethod
    def load(cls, path):
        with open(path) as f:
            data = json.load(f)
        m = cls(alpha=data['alpha'], min_df=data['min_df'])
        m.classes = data['classes']
        m.priors = data['priors']
        m.vocab = set(data['vocab'])
        m.vocab_size = data['vocab_size']
        m.class_total_words = {int(k): v for k, v in data['class_total_words'].items()}
        m.word_counts = {int(c): defaultdict(int, wc) for c, wc in data['word_counts'].items()}
        m._trained = True
        return m

    def top3(self, text):
        pred, probs = self.predict_with_proba(text)
        return sorted(probs.items(), key=lambda x: x[1], reverse=True)[:3]

### Train the model on the 200-row teaching sample

In [ ]:
texts = [r['text'] for r in sample_data]
labels = [int(r['category_encoded']) for r in sample_data]

model = MultinomialNB(alpha=1.0, min_df=1)  # min_df=1 for the small sample
model.fit(texts, labels)
print()
print('Vocabulary (first 10 tokens):', list(model.vocab)[:10])

### Make predictions — with all class probabilities

In [ ]:
def predict(text):
    pred, probs = model.predict_with_proba(text)
    print(f'Text      : "{text}"')
    print(f'Predicted : {CAT_DECODER[pred]}  (ID: {pred})')
    print(f'Confidence: {probs[pred]:.2%}')
    print('All probabilities:')
    for cid in sorted(probs.keys()):
        print(f'  {CAT_DECODER[cid]:22s} {probs[cid]:.2%}')
    print()

predict('wifi is slow in the library')
predict('canteen food quality is very bad')

---
## PART 5: EVALUATION METRICS

A small labelled test set exercised against the trained model. We report:

- **Accuracy** — fraction of correct predictions
- **Precision** — of what the model *predicted* for each category, how much was right
- **Recall** — of what *actually belonged* to each category, how much was found
- **F1** — harmonic mean of P and R (balanced measure)
- **Macro F1** — unweighted mean of per-class F1

In [ ]:
test_samples = [
    ('laptop charger not working', 0),          # IT Support
    ('food quality very bad in canteen', 9),    # Canteen
    ('bus is always late to campus', 5),        # Transport
    ('library AC not cooling today', 8),        # Library
    ('water leakage in my hostel room', 1),     # Hostels
    ('exam schedule not released yet', 2),      # Academics
]

correct = 0
per_class = {cname: {'tp': 0, 'fp': 0, 'fn': 0} for cname in CAT_DECODER.values()}
for text, true_label in test_samples:
    pred, probs = model.predict_with_proba(text)
    true_cat, pred_cat = CAT_DECODER[true_label], CAT_DECODER[pred]
    if pred == true_label:
        correct += 1
        per_class[pred_cat]['tp'] += 1
        print(f'OK  {text:44s} -> {pred_cat:22s}')
    else:
        per_class[pred_cat]['fp'] += 1
        per_class[true_cat]['fn'] += 1
        print(f'XX  {text:44s} -> {pred_cat:22s} (expected {true_cat})')

print(f'Accuracy: {correct}/{len(test_samples)} = {correct / len(test_samples) * 100:.1f}%')
print()
print(f'{"Category":20s} {"Precision":>10s} {"Recall":>10s} {"F1":>10s}')
print('-' * 52)
macro_f1_sum = 0
for cat in sorted(per_class.keys()):
    c = per_class[cat]
    p = c['tp'] / max(c['tp'] + c['fp'], 1)
    r = c['tp'] / max(c['tp'] + c['fn'], 1)
    f1 = 2 * p * r / max(p + r, 1)
    macro_f1_sum += f1
    print(f'{cat:20s} {p:>10.4f} {r:>10.4f} {f1:>10.4f}')
print(f'{"Macro F1":20s} {"":>10s} {"":>10s} {macro_f1_sum / len(per_class):>10.4f}')

---
## PART 6: SENTIMENT ANALYSIS — LEXICON-BASED

**Lexicons** — word → severity maps (negative words negative scores, positive words positive).

**State machine** — walk each token:

- `not/never/…` → flip polarity of the *next* sentiment word
- `very/extremely` → multiply the next sentiment word's impact

**Score** — average of detected sentiment word values.
**Label** — bucketed: `<= -0.5` Angry/Frustrated (Negative), `<= -0.1` Dissatisfied (Negative), `> 0.5` Appreciative (Positive), `> 0.1` Satisfied (Positive), else Neutral.

> Uses its own tokenizer (`re.findall`) that does **not** drop stopwords;
> `not`, `never`, etc. are treated as negations, not noise.

In [ ]:
NEGATION_WORDS = {'not', 'no', 'never', 'neither', 'nor', 'none', 'nothing',
                  'nobody', 'nowhere', 'cannot', "can't", "don't", "won't",
                  "wouldn't", "shouldn't", "couldn't", "isn't", "aren't",
                  "wasn't", "weren't", "haven't", "hasn't", "hadn't",
                  "didn't", "doesn't", "donot", "dont"}

INTENSIFIERS = {'very': 1.5, 'extremely': 2.0, 'really': 1.5, 'absolutely': 2.0,
                'completely': 1.5, 'totally': 1.5, 'highly': 1.5, 'strongly': 1.5,
                'utterly': 2.0, 'terribly': 1.5, 'so': 1.3, 'too': 1.3,
                'incredibly': 2.0, 'particularly': 1.3, 'exceptionally': 2.0}

NEGATIVE_WORDS = {
    # Severe (-3)
    'worst': -3, 'horrible': -3, 'terrible': -3, 'disgusting': -3,
    'unacceptable': -3, 'hopeless': -3, 'pathetic': -3, 'dreadful': -3,
    'outrageous': -3, 'inexcusable': -3, 'furious': -3,
    'harassment': -3, 'abuse': -3, 'robbery': -3, 'fraud': -3, 'scam': -3,
    'stolen': -3, 'cheated': -3, 'atrocious': -3, 'deplorable': -3,
    # Moderate (-2)
    'angry': -2, 'frustrated': -2, 'frustrating': -2, 'useless': -2,
    'ridiculous': -2, 'disappointed': -2, 'annoyed': -2, 'broken': -2,
    'damaged': -2, 'impossible': -2, 'unsafe': -2, 'dangerous': -2,
    'unhygienic': -2, 'malfunction': -2, 'stale': -2, 'smelly': -2,
    'fake': -2,
    # Mild (-1)
    'late': -1, 'delay': -1, 'problem': -1, 'issue': -1, 'trouble': -1,
    'difficult': -1, 'missing': -1, 'lost': -1, 'urgent': -1, 'critical': -1,
    'slow': -1, 'complaint': -1, 'leak': -1, 'leaking': -1, 'dirty': -1,
    'expired': -1, 'bad': -1, 'poor': -1, 'foul': -2,
}

POSITIVE_WORDS = {
    'excellent': 3, 'wonderful': 3, 'superb': 3, 'fantastic': 3, 'amazing': 3,
    'perfect': 3,
    'thank': 2, 'appreciate': 2, 'grateful': 2, 'great': 2, 'helpful': 2,
    'efficient': 2, 'satisfied': 2, 'happy': 2, 'pleased': 2, 'love': 2,
    'best': 2, 'nice': 2, 'friendly': 2, 'polite': 2, 'professional': 2,
    'good': 1, 'quick': 1, 'fast': 1, 'smooth': 1, 'easy': 1, 'clean': 1,
    'well': 1, 'better': 1, 'resolved': 1, 'solved': 1, 'fixed': 1,
    'help': 1, 'support': 1,
}

print('Lexicons loaded:')
print(f'  Negative: {len(NEGATIVE_WORDS)} words, Positive: {len(POSITIVE_WORDS)}', end='')
print(f', Negations: {len(NEGATION_WORDS)}, Intensifiers: {len(INTENSIFIERS)}')

In [ ]:
def analyze_sentiment(text):
    """Returns {label, sub_label, score, negations, total_sentiment_words}."""
    tokens = re.findall(r"[a-z]+'?[a-z]*", text.lower())
    score = 0.0
    word_count = 0
    negate_next = False
    intensify_next = 1.0
    negations_used = 0

    for token in tokens:
        if token in NEGATION_WORDS:
            negate_next = True
            negations_used += 1
            intensify_next = 1.0
            continue
        if token in INTENSIFIERS:
            intensify_next = INTENSIFIERS[token]
            continue
        multiplier = intensify_next * (-1.0 if negate_next else 1.0)
        if token in NEGATIVE_WORDS:
            score += NEGATIVE_WORDS[token] * multiplier
            word_count += 1
        elif token in POSITIVE_WORDS:
            score += POSITIVE_WORDS[token] * multiplier
            word_count += 1
        negate_next = False
        intensify_next = 1.0

    if word_count == 0:
        return {'label': 'Neutral', 'sub_label': 'Informational', 'score': 0.0,
                'negations': 0, 'total_sentiment_words': 0}

    avg = score / word_count
    if avg <= -0.5:
        label, category = 'Negative', 'Angry / Frustrated'
    elif avg <= -0.1:
        label, category = 'Negative', 'Dissatisfied'
    elif avg >= 0.5:
        label, category = 'Positive', 'Appreciative'
    elif avg >= 0.1:
        label, category = 'Positive', 'Satisfied'
    else:
        label, category = 'Neutral', 'Informational'

    return {'label': label, 'sub_label': category, 'score': round(avg, 3),
            'negations': negations_used, 'total_sentiment_words': word_count}

### Test sentiment analysis

In [ ]:
test_texts = [
    'Thank you for your help, really appreciate it',
    'The problem is still not fixed, very disappointed',
    'WiFi is not working in the library',
    'Absolutely ridiculous and unacceptable behavior',
    'Please look into the water leakage in my room',
    'I hate this college, worst experience ever, useless administration',
    'Great work by the maintenance team, very helpful',
    'exam schedule not released yet',
    'someone stole my laptop from the library',
]
print(f'{"Text":<52} {"Label":<10} {"Sub-label":<20} {"Score":>6}')
print('-' * 95)
for t in test_texts:
    r = analyze_sentiment(t)
    print(f'{t[:50]:<52} {r["label"]:<10} {r["sub_label"]:<20} {r["score"]:>6.3f}')

---
## PART 7: NEW WORD HANDLING IN SENTIMENT ANALYSIS

**Problem:** *"The food is atrocious"* — `atrocious` may be missing from the
lexicon, so a strong negative phrase is scored as neutral.

**Fix 1 (data-driven):** log unknown words to extend the lexicon.
**Fix 2 (rule-driven):** synonym fallback — treat unknown words as one of
their known synonyms.

In [ ]:
# Show the problem with words NOT in the lexicon.
missing = [w for w in ['epic', 'wondrous', 'deplorable', 'atrocious']
           if w not in NEGATIVE_WORDS and w not in POSITIVE_WORDS]
print('Words missing from both lexicons:', missing)
for w in ['wondrous', 'epic']:
    r = analyze_sentiment(f'The experience has been {w}')
    print(f'  "{w}" -> {r["label"]} (score {r["score"]})  <- ignored!')

### Fix 2 — Synonym mapping (small, explicit, deployable)

In [ ]:
# Known synonyms of the missing words above (a real entry point for growth)
SYNONYM_MAP = {
    'epic': 'wonderful', 'wondrous': 'wonderful', 'appalling': 'terrible',
    'atrocious': 'terrible', 'deplorable': 'terrible', 'dreadful': 'terrible',
    'useless': 'bad', 'brilliant': 'great', 'superb': 'excellent',
}

def analyze_sentiment_synonym(text):
    tokens = set(re.findall(r"[a-z]+'?[a-z]*", text.lower()))
    mapped = text
    for unknown in tokens:
        if unknown not in NEGATIVE_WORDS and unknown not in POSITIVE_WORDS:
            mapped = re.sub(r'\b' + unknown + r'\b',
                            SYNONYM_MAP.get(unknown, unknown), mapped, flags=re.I)
    return analyze_sentiment(mapped)

for text in ['The food was atrocious', 'Really deplorable conditions',
             'Absolutely wonderful experience']:
    r = analyze_sentiment(text)
    r2 = analyze_sentiment_synonym(text)
    print(f'{text[:50]:<52} base={r["label"]:<9} syn-map={r2["label"]:<9} (score {r2["score"]})')

---
## PART 8: COMPLETE SENTIMENT → PRIORITY PIPELINE

Production `ml/priority.py` converts sentiment + keywords into an explainable
`(priority, score, reason)`. Simplified, the rules are:

| Condition          | Points |
|-------------------|-------|
| sentiment Negative | +1 |
| sentiment <= -0.3  | +2 (strong negative) |
| risk keywords (`harass`, `unsafe`, `leak`…) | +2 |
| urgency keywords (`not working`, `urgent`, `asap`, `deadline`…) | +1 |
| appreciation words + positive sentiment | −1 |
| anomaly flag       | +2  |

Score → **High** (≥ 3), **Medium** (1–2), **Low** (≤ 0).

In [ ]:
RISK_PATTERNS = re.compile(
    r'\b(harass\w*|stole|theft|robbery|threaten\w*|unsafe|abuse|assault|'
    r'unhygienic|filthy|spoiled|rotten|stale\s+food|overcharg\w*|leak\w*|'
    r'no\s+(?:response|reply|action|update)|never\s+(?:fixed|resolved|addressed))\b',
    re.IGNORECASE
)
URGENT_PATTERNS = re.compile(
    r'\b(not\s+work(?:ing)?|doesn.?t\s+work|dont\s+work|wont\s+work|'
    r'broken|crack\w*|malfunction\w*|delay(?:s|ed)?|urgent|asap|'
    r'immediately|emergency|deadline|exam)\b',
    re.IGNORECASE
)
APPRECIATION = re.compile(
    r'\b(thank|appreciate|grateful|excellent|wonderful|amazing|great|'
    r'fixed|resolved|solved|helpful|happy|satisfied|delighted|impressed)\b',
    re.IGNORECASE
)

def compute_priority(text, sentiment, anomaly_flag=False):
    score = 0
    reasons = []
    if sentiment['label'] == 'Negative' and sentiment['score'] <= -0.3:
        score += 2
        reasons.append('strong negative tone')
    elif sentiment['label'] == 'Negative':
        score += 1
        reasons.append('negative tone')
    if RISK_PATTERNS.search(text):
        score += 2
        reasons.append('high-risk wording')
    if URGENT_PATTERNS.search(text):
        score += 1
        reasons.append('urgent issue')
    if anomaly_flag:
        score += 2
        reasons.append('anomaly flag')
    if sentiment['label'] == 'Positive' and APPRECIATION.search(text):
        score -= 1
        reasons.append('positive appreciation')
    score = max(score, 0)
    priority = 'High' if score >= 3 else ('Medium' if score >= 1 else 'Low')
    reason = ', '.join(reasons) if reasons else 'baseline'
    return priority, score, reason

def show_pipeline(text):
    print(f'Complaint      : "{text}"')
    pred, probs = model.predict_with_proba(text)
    print(f'  1. Category    : {CAT_DECODER[pred]}  (confidence {probs[pred]:.1%})')
    sent = analyze_sentiment(text)
    print(f'  2. Sentiment   : {sent["label"]} ({sent["sub_label"]}), score {sent["score"]}')
    prio, sc, reason = compute_priority(text, sent)
    print(f'  3. Priority    : {prio}  (score {sc} — {reason})')
    print()

show_pipeline('laptop charger not working, please fix it asap')
show_pipeline('thank you for fixing the canteen issue, great job')
show_pipeline('someone stole my belongings from the library')

---
## PART 9: ACTUAL MODEL PERFORMANCE — ACCURACY GRAPHS & LIVE LOG

This part reads the **real** `ml/training_log.json` written by production
`retrain.py` — the exact same file that powers the admin **Training** page
and **Reports** page. Nothing is hardcoded: retrain, re-run these cells,
and the numbers update.

> If matplotlib is installed the charts render inline; otherwise a plain-text
> fallback prints the same numbers.

In [ ]:
LOG_PATH = None
for _cand in ('ml/training_log.json', '../ml/training_log.json',
              '../../../ml/training_log.json',
              r'E:/Project-VI/workspace/ComplaintMgmtSystem/ml/training_log.json'):
    if os.path.isfile(_cand):
        LOG_PATH = _cand
        break
if LOG_PATH is None:
    print('WARNING: training_log.json not found — run ml/retrain.py first (continuing with empty log)')
    LOG = {}
else:
    with open(LOG_PATH) as f:
        LOG = json.load(f)
    print('Loaded training log:', LOG_PATH)
    print()
    print('=== MODEL PERFORMANCE SUMMARY ===')
    print('Last trained  :', LOG.get('last_trained'))
    print('Training rows :', LOG.get('train_samples'))
    print('Test rows     :', LOG.get('test_samples'))
    print('Accuracy      :', (str(LOG.get('accuracy')) + '%') if LOG.get('accuracy') is not None else 'N/A (too few test samples)')
    print('Macro F1      :', LOG.get('macro_f1'))
    hist = LOG.get('history', [])
    print('Retraining cycles:', len(hist))
    if hist:
        print('  first :', hist[0].get('date'), '->', hist[0].get('accuracy'))
        print('  latest:', hist[-1].get('date'), '->', hist[-1].get('accuracy'))

In [ ]:
try:
    from matplotlib import pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

hist = LOG.get('history', [])
if HAS_MPL and hist:
    dates = [h['date'][:10] for h in hist]
    accs = [h['accuracy'] for h in hist]
    plt.figure(figsize=(10, 4))
    plt.plot(dates, accs, marker='o', color='#d32f2f', linewidth=2, markersize=6)
    plt.ylim(0, 100)
    plt.ylabel('Accuracy (%)')
    plt.title('Classifier Accuracy Over Time')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Accuracy over time (text fallback):')
    for h in hist:
        a = h.get('accuracy')
        bar = ('#' * int(a // 5)) if a is not None else 'N/A'
        print(f'  {h["date"]:17s} {str(a) if a is not None else "N/A":>5}%  {bar}')

In [ ]:
per_class = LOG.get('per_class', [])
if HAS_MPL and per_class:
    cats = [m['category'] for m in per_class]
    f1s = [m['f1'] if m['f1'] is not None else 0 for m in per_class]
    precs = [m['precision'] if m['precision'] is not None else 0 for m in per_class]
    recs = [m['recall'] if m['recall'] is not None else 0 for m in per_class]
    samples = [m.get('samples', 0) for m in per_class]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x = range(len(cats)); w = 0.25
    ax1.bar([i - w for i in x], precs, w, label='Precision', color='#3b82f6')
    ax1.bar([i for i in x], recs, w, label='Recall', color='#10b981')
    ax1.bar([i + w for i in x], f1s, w, label='F1-Score', color='#d32f2f')
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(cats, rotation=45, ha='right', fontsize=9)
    ax1.set_ylim(0, 1.05)
    ax1.set_ylabel('Score')
    ax1.set_title('Per-Class Precision / Recall / F1')
    ax1.legend(fontsize=9)
    ax1.grid(axis='y', alpha=0.3)

    colors = ['#f59e0b' if s < 3 else '#3b82f6' for s in samples]
    ax2.bar(cats, samples, color=colors)
    ax2.set_xticks(range(len(cats)))
    ax2.set_xticklabels(cats, rotation=45, ha='right', fontsize=9)
    ax2.set_ylabel('Test Samples')
    ax2.set_title('Test Samples per Category')
    ax2.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Per-class metrics (text fallback):')
    for m in per_class:
        if m.get('f1') is not None:
            print(f'  {m["category"]:<24s} P={m["precision"]:.2f} R={m["recall"]:.2f} F1={m["f1"]:.2f} ({m.get("samples", 0)} samples)')
        else:
            print(f'  {m["category"]:<24s} N/A  (insufficient test samples)')

---
## PART 10: LIVE STATUS (MATCHES DASHBOARD)

Fresh read of `ml/training_log.json` — prints exactly what the admin
**Training** page shows: model version, train/test counts, accuracy, F1,
and a history trend.

In [ ]:
def load_live_log():
    for _cand in ('ml/training_log.json', '../ml/training_log.json',
                  '../../../ml/training_log.json',
                  r'E:/Project-VI/workspace/ComplaintMgmtSystem/ml/training_log.json'):
        if os.path.isfile(_cand):
            with open(_cand) as f:
                return json.load(f)
    return None

live = load_live_log()
if live is None:
    print('training_log.json not found — run ml/retrain.py first.')
else:
    print('=== LIVE MODEL STATUS (as shown on admin dashboard) ===')
    print(f'  Last trained : {live.get("last_trained")}')
    print(f'  Train samples: {live.get("train_samples")}')
    print(f'  Test samples : {live.get("test_samples")}')
    print(f'  Accuracy     : {live.get("accuracy")}%')
    print(f'  Macro F1     : {live.get("macro_f1")}')
    print(f'  Categories   : {len(live.get("per_class", []))}')
    hist = live.get('history', [])
    print(f'  Cycles logged: {len(hist)}')
    if hist:
        print(f'  Latest cycle : {hist[-1].get("date")} -> {hist[-1].get("accuracy")}%')
    print()
    print('History trend (last 8):')
    for h in hist[-8:]:
        a = h.get('accuracy')
        bar = ('#' * int(a // 5)) if a is not None else 'N/A'
        print(f'    {h["date"]:17s} {str(a) if a is not None else "N/A":>5}  {bar}')

### Closing notes

- **Why three metrics?** Accuracy alone hides per-category failures.
  Precision/Recall/F1 identify which categories the model reliably tags.
- **Where does the data come from?** `data/train_dataset.csv` (+ live DB rows
  in production retraining); `ml/training_log.json` summarizes each retrain.
- **Production parity** — this notebook's classifier, sentiment, and
  priority scoring mirror `ml/classifier.py`, `ml/sentiment.py`, and
  `ml/priority.py`; the only difference is the teaching-inline comments.

Retrain from the admin **Training** page or run `ml/retrain.py`, re-run
PART 9–10, and every number above updates automatically.